# SDXL Convert Phase 2-5 (TPU v5e-8, 377GB RAM)

Consumes Phase 1 kernel's output (`data_sdxl.pkl` + `images_sdxl/`) from `/kaggle/input/sdxl-convert-phase1-gpu/phase1_out/`.

Runs:
- Phase 2: `gen_quant_data_sdxl.py`
- Phase 3: `export_onnx_sdxl.py`
- Phase 4+5: `convert_all_sdxl.sh` (qairt converter + quantizer)

TPU sits idle — we're using this runtime for its 330GB system RAM + 96 CPU cores.

In [ ]:
CIVITAI_VERSION_ID = '__CIVITAI_VERSION_ID__'
MODEL_NAME         = '__MODEL_NAME__'
REALISTIC          = '__REALISTIC__' == 'true'
MIN_SOC            = '8gen4'
print(f'civitai={CIVITAI_VERSION_ID} name={MODEL_NAME} realistic={REALISTIC}')

In [ ]:
!free -h
!nproc
!df -h /kaggle/working /tmp
!cat /etc/os-release | head -5
!ldd --version | head -1

In [ ]:
# Locate Phase 1 output
import glob, os
candidates = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)
assert candidates, 'Phase 1 data_sdxl.pkl not found in /kaggle/input/ — did Phase 1 kernel complete?'
PHASE1_DIR = os.path.dirname(candidates[0])
print(f'Phase 1 dir: {PHASE1_DIR}')
!ls -lh $PHASE1_DIR

In [ ]:
# Civitai API key — injected by GH Actions from repo secret CIVITAI_API_KEY
import os
CIVITAI_API_KEY = '__CIVITAI_API_KEY__'
os.environ['CIVITAI_API_KEY'] = CIVITAI_API_KEY
os.environ['CIVITAI_VERSION_ID'] = CIVITAI_VERSION_ID
os.environ['MIN_SOC'] = MIN_SOC
os.environ['MODEL_NAME'] = MODEL_NAME
os.environ['MODEL_PATH'] = '/kaggle/working/model.safetensors'

In [ ]:
# Install tools + convertsdxl + QNN SDK
!apt-get install -y unzip zip curl time > /dev/null
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
import os
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

!rm -rf /tmp/convertsdxl.zip /tmp/convertsdxl_unzip
!curl -sL -o /tmp/convertsdxl.zip 'https://chino.icu/local-dream/convertsdxl.zip'
!mkdir -p /tmp/convertsdxl_unzip
!unzip -q /tmp/convertsdxl.zip -d /tmp/convertsdxl_unzip

import subprocess
root = subprocess.check_output("find /tmp/convertsdxl_unzip -maxdepth 3 -name 'export_sdxl.sh' -printf '%h\\n' | head -1", shell=True).decode().strip()
assert root, 'export_sdxl.sh not found'
NPUCONVERT_DIR = os.path.abspath(root)
os.environ['NPUCONVERT_DIR'] = NPUCONVERT_DIR
print(f'NPUCONVERT_DIR: {NPUCONVERT_DIR}')

In [ ]:
# QNN SDK
import os, subprocess
!mkdir -p /tmp/qnn_sdk
!cd /tmp/qnn_sdk && curl -sL -o qnn.zip 'https://apigwx-aws.qualcomm.com/qsc/public/v1/api/download/software/qualcomm_neural_processing_sdk/v2.28.0.241029.zip'
!cd /tmp/qnn_sdk && unzip -q qnn.zip
bin_file = subprocess.check_output('find /tmp/qnn_sdk -type f -name envsetup.sh -print -quit', shell=True).decode().strip()
assert bin_file
QNN_SDK_BIN      = os.path.dirname(bin_file)
QNN_SDK_ROOT_DIR = os.path.dirname(QNN_SDK_BIN)
os.environ['QNN_SDK_BIN']      = QNN_SDK_BIN
os.environ['QNN_SDK_ROOT_DIR'] = QNN_SDK_ROOT_DIR
print(f'QNN_SDK_ROOT: {QNN_SDK_ROOT_DIR}')

In [ ]:
# Re-download model (Phase 3 export_onnx needs PyTorch state dict)
import os
if not os.path.exists(os.environ['MODEL_PATH']) or os.path.getsize(os.environ['MODEL_PATH']) < int(1e9):
    !curl -L -H "Authorization: Bearer $CIVITAI_API_KEY" \
        -o "$MODEL_PATH" --fail --retry 5 --retry-delay 5 \
        "https://civitai.com/api/download/models/$CIVITAI_VERSION_ID"
!ls -lh $MODEL_PATH
assert os.path.getsize(os.environ['MODEL_PATH']) > int(1e9)

In [ ]:
# Python env
%cd $NPUCONVERT_DIR
!uv venv -p 3.10.17 --clear
!. .venv/bin/activate && uv sync

In [ ]:
# Copy Phase 1 output into convert dir
import os, shutil, glob
pkl = glob.glob('/kaggle/input/**/data_sdxl.pkl', recursive=True)[0]
p1_dir = os.path.dirname(pkl)
shutil.copy(pkl, f'{os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl')
if os.path.exists(f'{p1_dir}/images_sdxl'):
    dst = f'{os.environ["NPUCONVERT_DIR"]}/images_sdxl'
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(f'{p1_dir}/images_sdxl', dst)
!ls -lh $NPUCONVERT_DIR/data_sdxl.pkl
!ls $NPUCONVERT_DIR/images_sdxl | head -5

In [ ]:
# Start memory watcher
import subprocess, os
os.makedirs('/kaggle/working/logs', exist_ok=True)
watcher_sh = '''#!/bin/bash
echo "epoch,datetime,MemAvail_MB,TopRSS_MB,TopProc"
while true; do
  epoch=$(date +%s); dt=$(date -Iseconds)
  mem=$(awk '/MemAvailable:/{print int($2/1024)}' /proc/meminfo)
  line=$(ps -eo rss,comm --sort=-rss --no-headers 2>/dev/null | head -1)
  rss=$(echo "$line" | awk '{print int($1/1024)}')
  proc=$(echo "$line" | awk '{print $2}')
  echo "$epoch,$dt,$mem,$rss,$proc"
  sleep 10
done
'''
open('/tmp/mem_watch.sh','w').write(watcher_sh)
os.chmod('/tmp/mem_watch.sh', 0o755)
p = subprocess.Popen(['/tmp/mem_watch.sh'], stdout=open('/kaggle/working/logs/mem-trace.csv','w'), stderr=subprocess.DEVNULL)
open('/tmp/watcher.pid','w').write(str(p.pid))

In [ ]:
# Phase 2
import time
t0 = time.time()
%cd $NPUCONVERT_DIR
!. .venv/bin/activate && /usr/bin/time -v python gen_quant_data_sdxl.py \
    2>&1 | tee /kaggle/working/logs/phase2.log
print(f'Phase 2 elapsed: {int(time.time()-t0)}s')
!free -h

In [ ]:
# Phase 3
import time
t0 = time.time()
%cd $NPUCONVERT_DIR
!. .venv/bin/activate && /usr/bin/time -v python export_onnx_sdxl.py \
    --model_path "$MODEL_PATH" \
    2>&1 | tee /kaggle/working/logs/phase3.log
print(f'Phase 3 elapsed: {int(time.time()-t0)}s')
!free -h
!find . -name '*.onnx' -exec ls -lh {} \;
!find . -name 'weights.pb' -exec ls -lh {} \;

In [ ]:
# Backup Phase 3 ONNX output to /kaggle/working/phase3_backup
import os, shutil
bkp = '/kaggle/working/phase3_backup'
if os.path.exists(bkp): shutil.rmtree(bkp)
os.makedirs(bkp)
for d_ in ['clip_sdxl','clip2_sdxl','unet_sdxl','vae_decoder_sdxl','vae_encoder_sdxl']:
    src = f'{os.environ["NPUCONVERT_DIR"]}/{d_}'
    if os.path.exists(src):
        shutil.copytree(src, f'{bkp}/{d_}')
!ls $bkp
!du -sh $bkp/*
!find $NPUCONVERT_DIR -maxdepth 2 -name 'input_list_*.txt' -exec cp {} $bkp/ \;
!ls $bkp/*.txt 2>/dev/null || true

In [ ]:
# Patch convert_all_sdxl.sh paths
import os
!sed -i "s|QNN_SDK_ROOT=/data/qairt/[0-9.]*|QNN_SDK_ROOT=$QNN_SDK_ROOT_DIR|" $NPUCONVERT_DIR/scripts/convert_all_sdxl.sh
!grep -nE 'QNN_SDK_ROOT|^cd |source envsetup' $NPUCONVERT_DIR/scripts/convert_all_sdxl.sh

In [ ]:
# Phase 4-5 — QNN convert + quantize
import time
t0 = time.time()
%cd $NPUCONVERT_DIR
!. .venv/bin/activate && /usr/bin/time -v bash scripts/convert_all_sdxl.sh \
    --min_soc $MIN_SOC \
    2>&1 | tee /kaggle/working/logs/phase45.log
print(f'Phase 4-5 elapsed: {int(time.time()-t0)}s')
!free -h

In [ ]:
# Package output zip
import os
%cd $NPUCONVERT_DIR
out_dir = f'output/qnn_models_sdxl_{os.environ["MIN_SOC"]}'
if os.path.isdir(out_dir):
    !touch {out_dir}/SDXL
    zip_name = f'{os.environ["MODEL_NAME"]}_qnn2.28_{os.environ["MIN_SOC"]}.zip'
    !zip -r /kaggle/working/{zip_name} {out_dir}
    !ls -lh /kaggle/working/{zip_name}
else:
    print(f'{out_dir} missing — Phase 4-5 did not complete')
# stop watcher
try:
    os.kill(int(open('/tmp/watcher.pid').read()), 9)
except Exception: pass

In [ ]:
# Summary
import pandas as pd, os
if os.path.exists('/kaggle/working/logs/mem-trace.csv'):
    df = pd.read_csv('/kaggle/working/logs/mem-trace.csv')
    print(f'Samples: {len(df)}')
    if len(df):
        print('5 lowest MemAvail (MB):')
        print(df.nsmallest(5,'MemAvail_MB')[['datetime','MemAvail_MB','TopRSS_MB','TopProc']].to_string())
        print('5 highest TopRSS (MB):')
        print(df.nlargest(5,'TopRSS_MB')[['datetime','TopRSS_MB','TopProc']].to_string())
!ls -lh /kaggle/working/